In [ ]:
# ==========================================
# CT26 TASK 1 — NEMOTRON RERANKER 1B FULL FINE-TUNING | TOP-10 ONLY
#
# Model:
#   nvidia/llama-nemotron-rerank-1b-v2
#
# Design:
# - Uses your trained Qwen3-Embedding-8B LoRA retriever artifacts.
# - Candidate depth: top-10 only.
# - Training group: 1 gold + 9 hardest negatives from Qwen retriever top-10.
# - Nemotron score: raw sequence-classification relevance logit.
# - Loss: listwise softmax CE over 10 candidates.
# - FULL FINE-TUNING: no LoRA, all model params trainable.
# - Max epochs: 3.
# - Early stopping patience: 1.
# - Gradient checkpointing enabled.
# - Autotunes largest safe training group batch size.
# - Saves full HF model checkpoints, not PEFT adapters.
# ==========================================


# ==========================================
# 1. MOUNT DRIVE + DEFINE SHARED PATHS
# ==========================================

import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive"

SHARED_RETRIEVER_ROOT = os.path.join(
    DRIVE_ROOT,
    "ct26_qwen3_embedding_8b_lora",
)

SHARED_RERANKER_ROOT = os.path.join(
    DRIVE_ROOT,
    "ct26_nemotron1b_full_finetune_listwise_top10",
)

assert os.path.isdir(SHARED_RETRIEVER_ROOT), (
    f"Could not find retriever folder:\n{SHARED_RETRIEVER_ROOT}"
)

os.makedirs(SHARED_RERANKER_ROOT, exist_ok=True)

BEST_RETRIEVER_DIR = os.path.join(
    SHARED_RETRIEVER_ROOT,
    "best_qwen3_8b_lora_sentence_transformer",
)

RETRIEVER_ARTIFACTS_DIR = os.path.join(
    SHARED_RETRIEVER_ROOT,
    "retrieval_artifacts",
)

assert os.path.isdir(BEST_RETRIEVER_DIR), f"Missing retriever model dir: {BEST_RETRIEVER_DIR}"
assert os.path.isdir(RETRIEVER_ARTIFACTS_DIR), f"Missing retriever artifacts dir: {RETRIEVER_ARTIFACTS_DIR}"

STAGE2_ROOT = SHARED_RERANKER_ROOT

CHECKPOINT_DIR = os.path.join(STAGE2_ROOT, "trainer_checkpoints")
BEST_RERANKER_DIR = os.path.join(STAGE2_ROOT, "best_nemotron1b_full_finetuned")
CACHE_DIR = os.path.join(STAGE2_ROOT, "cache")
EVAL_DIR = os.path.join(STAGE2_ROOT, "eval")

for p in [STAGE2_ROOT, CHECKPOINT_DIR, BEST_RERANKER_DIR, CACHE_DIR, EVAL_DIR]:
    os.makedirs(p, exist_ok=True)

# Reuse top-10 cache from your Qwen8B reranker experiment if available.
EXISTING_QWEN_TOP10_CACHE_DIR = os.path.join(
    DRIVE_ROOT,
    "ct26_qwen3_reranker8b_lora_listwise_top10",
    "cache",
)

print("Retriever root:", SHARED_RETRIEVER_ROOT)
print("Full FT Nemotron root:", SHARED_RERANKER_ROOT)
print("Best retriever dir:", BEST_RETRIEVER_DIR)
print("Retriever artifacts dir:", RETRIEVER_ARTIFACTS_DIR)
print("Best full FT model will save to:", BEST_RERANKER_DIR)
print("Optional existing Qwen top-10 cache dir:", EXISTING_QWEN_TOP10_CACHE_DIR)


# ==========================================
# 2. INSTALL
# ==========================================

import sys
import subprocess

INSTALLATION_OK = False

def run(cmd):
    print(f"\n$ {cmd}", flush=True)
    result = subprocess.run(cmd, shell=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

def run_optional(cmd):
    print(f"\n$ {cmd}", flush=True)
    result = subprocess.run(cmd, shell=True, text=True)
    if result.returncode != 0:
        print(f"Optional command failed; continuing:\n{cmd}")
    return result.returncode == 0

# Keep Colab's existing torch.
run(f"""{sys.executable} -m pip install -U \
"transformers>=4.51.0" \
"sentence-transformers>=5.0.0" \
"datasets>=2.19.0" \
"accelerate>=0.30.0" \
safetensors tqdm scikit-learn packaging ninja""")

# Optional helper; not required.
run_optional(f"{sys.executable} -m pip install -U torchao --index-url https://download.pytorch.org/whl/cu128")

# Optional FlashAttention wheel for common Colab A100:
# Python 3.12 + torch 2.10 + CUDA 12.8.
FLASH_ATTN_WHEEL = "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
run_optional(f'{sys.executable} -m pip install -U "{FLASH_ATTN_WHEEL}"')

import torch
import transformers
from transformers.utils import is_flash_attn_2_available

print("\n===== VERIFICATION =====")
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA used by torch:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None)
print("transformers:", transformers.__version__)
print("FA2 available to Transformers:", is_flash_attn_2_available())

assert torch.cuda.is_available(), "CUDA is not available"
assert torch.cuda.is_bf16_supported(), "BF16 is not supported"

INSTALLATION_OK = True
print("\nINSTALLATION_OK = True")


# ==========================================
# 3. IMPORTS + SAFETY CHECKS
# ==========================================

assert INSTALLATION_OK is True, "Installation failed. Do not run training."

import gc
import json
import gzip
import math
import time
import random
import shutil
from typing import Any

import numpy as np
import torch.nn.functional as F

from datasets import load_dataset
from tqdm.auto import tqdm
from torch.utils.data import Dataset as TorchDataset, DataLoader

from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

assert torch.cuda.is_available(), "CUDA is required."
assert torch.cuda.is_bf16_supported(), "BF16 is required/recommended for A100."

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Best set before Python process starts, but harmless here.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", torch.cuda.is_bf16_supported())
print("Stage-2 root:", STAGE2_ROOT)
print("FlashAttention 2 available:", is_flash_attn_2_available())


# ==========================================
# 4. CONFIG
# ==========================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

HF_TOKEN = os.environ.get("HF_TOKEN", None)

DATASET_NAME = "sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims"
LANGUAGES = ["en", "fr", "de"]

NEMOTRON_MODEL_NAME = "nvidia/llama-nemotron-rerank-1b-v2"

TOPK = 10
TRAIN_RETRIEVAL_TOPK = TOPK
DEV_RERANK_K = TOPK
LISTWISE_GROUP_SIZE = TOPK
assert LISTWISE_GROUP_SIZE == 10

RETRIEVER_TASK_INSTRUCTION = (
    "Given a scientific web claim, retrieve the title and abstract of the "
    "scientific publication that is the source or best evidence for the claim."
)

NEMOTRON_QUERY_VARIANT = "raw"  # "raw" keeps comparability with your Nemotron LoRA run.
NEMOTRON_MAX_LENGTH = 512
MODEL_DTYPE = torch.bfloat16
USE_FLASH_ATTN = is_flash_attn_2_available()

# Full FT schedule.
MAX_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 1

# Autotune training batch.
AUTOTUNE_TRAIN_BATCH_SIZE = True

# Candidate group batch sizes to try.
# Each group = 10 query-doc pairs.
TRAIN_GROUP_BATCH_SIZE_CANDIDATES = [8, 6, 4, 3, 2, 1]

# Target effective group batch per optimizer step.
# If autotune picks microbatch=1, grad accum=32.
# If microbatch=2, grad accum=16, etc.
TARGET_EFFECTIVE_GROUPS_PER_UPDATE = 32

# Full FT usually needs smaller LR than LoRA.
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
LOGGING_STEPS = 50

# Full checkpoints are large.
SAVE_EVERY_OPT_STEPS = 0
DELETE_NON_BEST_EPOCH_CHECKPOINTS = True

FORCE_REBUILD_TRAIN_CANDIDATES = False
FORCE_REBUILD_DEV_CANDIDATES = False
FORCE_REBUILD_TRAIN_GROUPS = False

RUN_PRETRAIN_EVAL = False

# Safe eval fallback. Do not start at 1024; it OOMed previously.
PAIR_EVAL_BATCH_CANDIDATES = [128, 96, 64, 48, 32, 24, 16, 8, 4, 2, 1]

FUSION_ALPHAS = [round(float(x), 3) for x in np.arange(0.0, 1.0001, 0.025)]

TRAIN_CANDIDATES_GZ = os.path.join(CACHE_DIR, f"train_candidates_top{TOPK}.json.gz")
DEV_CANDIDATES_GZ = os.path.join(CACHE_DIR, f"dev_candidates_top{TOPK}.json.gz")
TRAIN_GROUPS_GZ = os.path.join(CACHE_DIR, f"train_listwise_groups_top{TOPK}.json.gz")

TRAINING_HISTORY_PATH = os.path.join(EVAL_DIR, "training_history.json")
FINAL_ALPHA_SUMMARY_PATH = os.path.join(EVAL_DIR, f"best_alpha_sweep_summary_top{TOPK}.json")
FINAL_METRICS_PATH = os.path.join(EVAL_DIR, f"best_nemotron1b_full_finetuned_top{TOPK}_metrics.json")
META_PATH = os.path.join(STAGE2_ROOT, "meta.json")

DOC_EMB_PATH = os.path.join(RETRIEVER_ARTIFACTS_DIR, "qwen3_8b_lora_doc_embeddings.npy")
DOC_IDS_PATH = os.path.join(RETRIEVER_ARTIFACTS_DIR, "doc_ids_list.json")
DOC_RAW_PATH = os.path.join(RETRIEVER_ARTIFACTS_DIR, "doc_raw.json.gz")

assert os.path.isfile(DOC_EMB_PATH), f"Missing doc embeddings: {DOC_EMB_PATH}"
assert os.path.isfile(DOC_IDS_PATH), f"Missing doc IDs: {DOC_IDS_PATH}"
assert os.path.isfile(DOC_RAW_PATH), f"Missing doc raw: {DOC_RAW_PATH}"


# ==========================================
# 5. HELPERS
# ==========================================

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def print_gpu_memory(label: str):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        max_alloc = torch.cuda.max_memory_allocated() / 1024**3
        print(
            f"[GPU MEM] {label}: "
            f"allocated={allocated:.2f}GB reserved={reserved:.2f}GB max_allocated={max_alloc:.2f}GB"
        )

def save_json(path: str, obj: Any):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json_gz(path: str, obj: Any):
    with gzip.open(path, "wt", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

def load_json_gz(path: str):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return json.load(f)

def normalize_ws(text: Any) -> str:
    return " ".join(str(text).split())

def dataset_kwargs():
    kwargs = {}
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    return kwargs

def load_ct26_split(lang: str, split: str):
    return load_dataset(DATASET_NAME, lang, split=split, **dataset_kwargs())

def get_detailed_instruct(task_description: str, query: str) -> str:
    return f"Instruct: {task_description}\nQuery:{query}"

def zscore(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)

def compute_metrics_from_ranks(ranks):
    ranks = np.asarray(ranks)
    return {
        "MRR@1": float(np.mean([1.0 / r if r <= 1 else 0.0 for r in ranks])),
        "MRR@5": float(np.mean([1.0 / r if r <= 5 else 0.0 for r in ranks])),
        "MRR@10": float(np.mean([1.0 / r if r <= 10 else 0.0 for r in ranks])),
        "Recall@5": float((ranks <= 5).mean()),
        "Recall@10": float((ranks <= 10).mean()),
    }

def print_multilingual_metrics(title, metrics):
    print(f"\n===== {title} =====")
    for lang in LANGUAGES:
        print(f"--- {lang.upper()} ---")
        print(f"  MRR@1     : {metrics[f'{lang}_mrr@1']:.4f}")
        print(f"  MRR@5     : {metrics[f'{lang}_mrr@5']:.4f}")
        print(f"  MRR@10    : {metrics[f'{lang}_mrr@10']:.4f}")
        print(f"  Recall@5  : {metrics[f'{lang}_recall@5']:.4f}")
        print(f"  Recall@10 : {metrics[f'{lang}_recall@10']:.4f}")
    print("===== MULTILINGUAL AVG =====")
    print(f"  MRR@1     : {metrics['multilingual_avg_mrr@1']:.4f}")
    print(f"  MRR@5     : {metrics['multilingual_avg_mrr@5']:.4f}")
    print(f"  MRR@10    : {metrics['multilingual_avg_mrr@10']:.4f}")
    print(f"  Recall@5  : {metrics['multilingual_avg_recall@5']:.4f}")
    print(f"  Recall@10 : {metrics['multilingual_avg_recall@10']:.4f}")

def get_topk_from_scores(scores, k):
    k = min(k, scores.shape[1])
    part = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]
    part_scores = np.take_along_axis(scores, part, axis=1)
    order = np.argsort(-part_scores, axis=1)
    top_idx = np.take_along_axis(part, order, axis=1)
    top_scores = np.take_along_axis(part_scores, order, axis=1)
    return top_idx, top_scores

def batched_retrieve_topk(query_texts, retriever, doc_matrix, topk, batch_size=32):
    all_topk_idx = []
    all_topk_scores = []

    for start in tqdm(range(0, len(query_texts), batch_size), desc=f"Qwen retriever top-{topk}"):
        batch_queries = query_texts[start:start + batch_size]

        with torch.inference_mode():
            q_emb = retriever.encode(
                batch_queries,
                batch_size=batch_size,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            ).astype(np.float32)

        scores = q_emb @ doc_matrix.T
        batch_topk_idx, batch_topk_scores = get_topk_from_scores(scores, topk)

        all_topk_idx.extend(batch_topk_idx.tolist())
        all_topk_scores.extend(batch_topk_scores.astype(np.float32).tolist())

        del q_emb, scores, batch_topk_idx, batch_topk_scores
        cleanup_cuda()

    return all_topk_idx, all_topk_scores


# ==========================================
# 6. NEMOTRON PROMPT + SCORING
# ==========================================

def nemotron_query_text(query):
    query = normalize_ws(query)

    if NEMOTRON_QUERY_VARIANT == "raw":
        return query

    if NEMOTRON_QUERY_VARIANT == "instructed":
        return (
            f"{RETRIEVER_TASK_INSTRUCTION}\n\n"
            f"Scientific web claim: {query}"
        )

    raise ValueError(f"Unknown NEMOTRON_QUERY_VARIANT: {NEMOTRON_QUERY_VARIANT}")

def make_nemotron_text(query, doc):
    q = nemotron_query_text(query)
    return f"question:{q} \n \n passage:{doc}"

def estimate_pair_len(pair):
    q, d = pair
    return len(str(q).split()) + len(str(d).split())

def forward_nemotron_scores(model, batch):
    outputs = model(**batch)
    logits = outputs.logits

    if logits.ndim == 2 and logits.shape[1] == 1:
        return logits[:, 0]

    if logits.ndim == 2 and logits.shape[1] == 2:
        return logits[:, 1] - logits[:, 0]

    return logits.view(logits.shape[0], -1)[:, 0]

def tokenize_nemotron_texts(tokenizer, texts, max_length):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
        pad_to_multiple_of=8,
    )

def score_flat_pairs_nemotron_sorted(
    model,
    tokenizer,
    pairs,
    batch_size,
    device,
    desc,
):
    n = len(pairs)
    if n == 0:
        return np.zeros((0,), dtype=np.float32)

    model.eval()

    lengths = np.asarray([estimate_pair_len(p) for p in pairs], dtype=np.int32)
    order = np.argsort(lengths)
    sorted_pairs = [pairs[i] for i in order]

    sorted_scores = np.empty(n, dtype=np.float32)

    with torch.inference_mode():
        for start in tqdm(range(0, n, batch_size), desc=desc):
            batch_pairs = sorted_pairs[start:start + batch_size]
            texts = [make_nemotron_text(q, d) for q, d in batch_pairs]

            batch = tokenize_nemotron_texts(
                tokenizer=tokenizer,
                texts=texts,
                max_length=NEMOTRON_MAX_LENGTH,
            )
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            with torch.autocast(device_type="cuda", dtype=MODEL_DTYPE, enabled=True):
                batch_scores = forward_nemotron_scores(model, batch)

            vals = batch_scores.detach().float().cpu().numpy().astype(np.float32)
            sorted_scores[start:start + len(vals)] = vals

            del batch, batch_scores, vals
            cleanup_cuda()

    unsorted_scores = np.empty(n, dtype=np.float32)
    unsorted_scores[order] = sorted_scores
    return unsorted_scores

def score_flat_pairs_with_fallback(model, tokenizer, pairs, device, desc_prefix):
    last_error = None

    for bs in PAIR_EVAL_BATCH_CANDIDATES:
        try:
            print(f"Trying eval batch size {bs} for {desc_prefix}")
            cleanup_cuda()
            return score_flat_pairs_nemotron_sorted(
                model=model,
                tokenizer=tokenizer,
                pairs=pairs,
                batch_size=bs,
                device=device,
                desc=f"{desc_prefix} bs={bs}",
            ), bs

        except RuntimeError as e:
            last_error = e
            if "out of memory" in str(e).lower():
                print(f"OOM at eval batch size {bs}. Trying smaller...")
                cleanup_cuda()
            else:
                raise

    raise RuntimeError(f"Could not score {desc_prefix}; last error: {repr(last_error)}")


# ==========================================
# 7. LOAD RETRIEVER ARTIFACTS
# ==========================================

print("\nLoading Qwen retriever artifacts...")
doc_matrix = np.load(DOC_EMB_PATH).astype(np.float32)
doc_ids_list = load_json(DOC_IDS_PATH)
doc_raw = load_json_gz(DOC_RAW_PATH)

doc_ids_list = [str(x) for x in doc_ids_list]
doc_raw = {str(k): v for k, v in doc_raw.items()}
docid_to_idx = {did: idx for idx, did in enumerate(doc_ids_list)}

print("Doc matrix shape:", doc_matrix.shape)
print("Documents:", len(doc_ids_list))


# ==========================================
# 8. LOAD TRAIN/DEV SPLITS
# ==========================================

print("\nLoading train/dev splits...")
combined_train = []
dev_by_lang_raw = {}

for lang in LANGUAGES:
    train_split = load_ct26_split(lang, "train")
    dev_split = load_ct26_split(lang, "dev")

    for item in train_split:
        row = dict(item)
        row["lang"] = lang
        combined_train.append(row)

    dev_by_lang_raw[lang] = [dict(x) for x in dev_split]

    print(f"{lang}: train={len(train_split)}, dev={len(dev_split)}")

print("Total train rows:", len(combined_train))


# ==========================================
# 9. REUSE OR BUILD TRAIN/DEV CANDIDATES
# ==========================================

def maybe_copy_existing_cache(filename, dest_path):
    src_path = os.path.join(EXISTING_QWEN_TOP10_CACHE_DIR, filename)
    if os.path.isfile(dest_path):
        return False
    if os.path.isfile(src_path):
        shutil.copy2(src_path, dest_path)
        print(f"Copied cache:\n  from: {src_path}\n  to:   {dest_path}")
        return True
    return False

maybe_copy_existing_cache(f"train_candidates_top{TOPK}.json.gz", TRAIN_CANDIDATES_GZ)
maybe_copy_existing_cache(f"dev_candidates_top{TOPK}.json.gz", DEV_CANDIDATES_GZ)
maybe_copy_existing_cache(f"train_listwise_groups_top{TOPK}.json.gz", TRAIN_GROUPS_GZ)

def build_candidates_for_rows(rows, retriever, doc_matrix, doc_ids_list, doc_raw, topk, batch_size, split_name):
    queries_raw = [normalize_ws(x["text"]) for x in rows]
    queries_for_retriever = [
        get_detailed_instruct(RETRIEVER_TASK_INSTRUCTION, q)
        for q in queries_raw
    ]
    gold_ids = [str(x["pubkey"]) for x in rows]

    topk_idx, topk_scores = batched_retrieve_topk(
        query_texts=queries_for_retriever,
        retriever=retriever,
        doc_matrix=doc_matrix,
        topk=topk,
        batch_size=batch_size,
    )

    samples = []

    for i, gold_id in enumerate(tqdm(gold_ids, desc=f"Build {split_name} samples")):
        candidate_ids = [doc_ids_list[idx] for idx in topk_idx[i]]
        candidate_texts = [doc_raw[did] for did in candidate_ids]

        samples.append({
            "query": queries_raw[i],
            "query_with_retriever_instruction": queries_for_retriever[i],
            "gold_id": gold_id,
            "candidate_ids": candidate_ids,
            "candidate_texts": candidate_texts,
            "dense_scores": topk_scores[i],
            "retriever_scores": topk_scores[i],
            "lang": rows[i].get("lang", None),
        })

    return samples

need_train_candidates = FORCE_REBUILD_TRAIN_CANDIDATES or not os.path.isfile(TRAIN_CANDIDATES_GZ)
need_dev_candidates = FORCE_REBUILD_DEV_CANDIDATES or not os.path.isfile(DEV_CANDIDATES_GZ)

if need_train_candidates or need_dev_candidates:
    print("\nLoading Qwen LoRA retriever for candidate generation...")
    print_gpu_memory("before retriever load")

    retriever_kwargs = {
        "attn_implementation": "flash_attention_2" if USE_FLASH_ATTN else "sdpa",
        "device_map": {"": 0},
    }

    try:
        retriever_kwargs["dtype"] = torch.bfloat16
        retriever = SentenceTransformer(
            BEST_RETRIEVER_DIR,
            model_kwargs=retriever_kwargs,
            processor_kwargs={"padding_side": "left"},
        )
    except TypeError:
        retriever_kwargs.pop("dtype", None)
        retriever_kwargs["torch_dtype"] = torch.bfloat16
        retriever = SentenceTransformer(
            BEST_RETRIEVER_DIR,
            model_kwargs=retriever_kwargs,
            tokenizer_kwargs={"padding_side": "left"},
        )

    retriever.max_seq_length = 512
    retriever.tokenizer.padding_side = "left"
    retriever.eval()

    print_gpu_memory("after retriever load")

    if need_train_candidates:
        print("\nBuilding train top-10 candidates...")
        train_candidates = build_candidates_for_rows(
            rows=combined_train,
            retriever=retriever,
            doc_matrix=doc_matrix,
            doc_ids_list=doc_ids_list,
            doc_raw=doc_raw,
            topk=TRAIN_RETRIEVAL_TOPK,
            batch_size=32,
            split_name="train",
        )
        save_json_gz(TRAIN_CANDIDATES_GZ, train_candidates)
        print("Saved train candidates:", TRAIN_CANDIDATES_GZ)

    if need_dev_candidates:
        print("\nBuilding dev top-10 candidates...")
        dev_candidates_by_lang = {}

        for lang in LANGUAGES:
            dev_candidates_by_lang[lang] = build_candidates_for_rows(
                rows=dev_by_lang_raw[lang],
                retriever=retriever,
                doc_matrix=doc_matrix,
                doc_ids_list=doc_ids_list,
                doc_raw=doc_raw,
                topk=DEV_RERANK_K,
                batch_size=32,
                split_name=f"dev-{lang}",
            )

        save_json_gz(DEV_CANDIDATES_GZ, dev_candidates_by_lang)
        print("Saved dev candidates:", DEV_CANDIDATES_GZ)

    print("\nUnloading retriever before loading Nemotron...")
    try:
        retriever.cpu()
    except Exception:
        pass

    del retriever
    cleanup_cuda()
    print_gpu_memory("after retriever unload")

train_candidates = load_json_gz(TRAIN_CANDIDATES_GZ)
dev_candidates_by_lang = load_json_gz(DEV_CANDIDATES_GZ)

print("Train candidates:", len(train_candidates))
for lang in LANGUAGES:
    print(f"Dev candidates {lang}:", len(dev_candidates_by_lang[lang]))


# ==========================================
# 10. BUILD LISTWISE TRAIN GROUPS
# ==========================================

def build_train_listwise_groups_from_candidates(samples, doc_raw, group_size=10, seed=42):
    groups = []
    skipped = 0
    injected_gold = 0
    gold_already_present = 0

    for i, sample in enumerate(tqdm(samples, desc="Building listwise train groups")):
        q = normalize_ws(sample["query"])
        gold_id = str(sample["gold_id"])
        candidate_ids = [str(x) for x in sample["candidate_ids"]]

        if gold_id not in doc_raw:
            skipped += 1
            continue

        wrong_ids = [did for did in candidate_ids if did != gold_id]

        if len(wrong_ids) < group_size - 1:
            skipped += 1
            continue

        if gold_id in candidate_ids:
            group_ids = candidate_ids[:group_size]
            if gold_id not in group_ids:
                group_ids = [gold_id] + wrong_ids[:group_size - 1]
                injected_gold += 1
            else:
                gold_already_present += 1
        else:
            group_ids = [gold_id] + wrong_ids[:group_size - 1]
            injected_gold += 1

        group_ids = group_ids[:group_size]
        if gold_id not in group_ids:
            group_ids[-1] = gold_id

        rng = np.random.default_rng(seed + i)
        perm = rng.permutation(group_size)
        shuffled_ids = [group_ids[j] for j in perm]
        positive_index = shuffled_ids.index(gold_id)

        groups.append({
            "query": q,
            "candidate_ids": shuffled_ids,
            "candidate_texts": [doc_raw[did] for did in shuffled_ids],
            "positive_index": positive_index,
            "gold_id": gold_id,
            "lang": sample.get("lang", None),
        })

    print("Train groups:", len(groups))
    print("Skipped:", skipped)
    print("Gold already in retrieved group:", gold_already_present)
    print("Gold injected:", injected_gold)

    return groups

if FORCE_REBUILD_TRAIN_GROUPS or not os.path.isfile(TRAIN_GROUPS_GZ):
    train_groups = build_train_listwise_groups_from_candidates(
        samples=train_candidates,
        doc_raw=doc_raw,
        group_size=LISTWISE_GROUP_SIZE,
        seed=SEED,
    )
    save_json_gz(TRAIN_GROUPS_GZ, train_groups)
    print("Saved train groups:", TRAIN_GROUPS_GZ)
else:
    train_groups = load_json_gz(TRAIN_GROUPS_GZ)
    print("Loaded train groups:", len(train_groups))

assert len(train_groups) > 0, "No train groups were built."


# ==========================================
# 11. DATASET + COLLATOR
# ==========================================

class ListwiseNemotronDataset(TorchDataset):
    def __init__(self, groups):
        self.groups = groups

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        return self.groups[idx]

class ListwiseNemotronCollator:
    def __init__(self, tokenizer, group_size, max_length):
        self.tokenizer = tokenizer
        self.group_size = group_size
        self.max_length = max_length

    def __call__(self, batch):
        texts = []
        positive_indices = []

        for item in batch:
            assert len(item["candidate_texts"]) == self.group_size
            q = item["query"]
            for d in item["candidate_texts"]:
                texts.append(make_nemotron_text(q, d))
            positive_indices.append(item["positive_index"])

        features = tokenize_nemotron_texts(
            tokenizer=self.tokenizer,
            texts=texts,
            max_length=self.max_length,
        )

        features["positive_indices"] = torch.tensor(positive_indices, dtype=torch.long)
        features["num_groups"] = len(batch)
        return features


# ==========================================
# 12. LOAD NEMOTRON FULL MODEL
# ==========================================

print("\nLoading Nemotron reranker for FULL fine-tuning...")
print_gpu_memory("before Nemotron load")

tokenizer = AutoTokenizer.from_pretrained(
    NEMOTRON_MODEL_NAME,
    padding_side="left",
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_load_kwargs = {
    "trust_remote_code": True,
    "device_map": {"": 0},
}

# Transformers 5 prefers dtype; older versions prefer torch_dtype.
try:
    model_load_kwargs["dtype"] = MODEL_DTYPE
    model_load_kwargs["attn_implementation"] = "flash_attention_2" if USE_FLASH_ATTN else "sdpa"
    model = AutoModelForSequenceClassification.from_pretrained(
        NEMOTRON_MODEL_NAME,
        **model_load_kwargs,
    )
except TypeError:
    model_load_kwargs.pop("dtype", None)
    model_load_kwargs["torch_dtype"] = MODEL_DTYPE
    model_load_kwargs["attn_implementation"] = "flash_attention_2" if USE_FLASH_ATTN else "sdpa"
    model = AutoModelForSequenceClassification.from_pretrained(
        NEMOTRON_MODEL_NAME,
        **model_load_kwargs,
    )

if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.pad_token_id

try:
    model.config.use_cache = False
except Exception:
    pass

# Gradient checkpointing is important for full FT.
try:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    print("Enabled gradient checkpointing with use_reentrant=False.")
except TypeError:
    model.gradient_checkpointing_enable()
    print("Enabled gradient checkpointing.")
except Exception as e:
    print("Could not enable gradient checkpointing:", repr(e))

# Full FT: ensure everything trainable.
for p in model.parameters():
    p.requires_grad_(True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

device = torch.device("cuda")
model.train()

print_gpu_memory("after Nemotron full model load")


# ==========================================
# 13. AUTOTUNE TRAIN GROUP BATCH SIZE
# ==========================================

train_dataset = ListwiseNemotronDataset(train_groups)
base_collator = ListwiseNemotronCollator(
    tokenizer=tokenizer,
    group_size=LISTWISE_GROUP_SIZE,
    max_length=NEMOTRON_MAX_LENGTH,
)

def group_length_estimate(group):
    q = group["query"]
    return max(len(str(d).split()) + len(str(q).split()) for d in group["candidate_texts"])

def make_probe_groups(groups, n):
    # Use longest examples for safer autotuning.
    sorted_groups = sorted(groups, key=group_length_estimate, reverse=True)
    return sorted_groups[:n]

def try_train_microbatch(model, collator, groups, bs):
    probe_groups = make_probe_groups(groups, bs)
    batch = collator(probe_groups)

    positive_indices = batch.pop("positive_indices").to(device)
    num_groups = int(batch.pop("num_groups"))

    batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

    model.train()
    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()

    with torch.autocast(device_type="cuda", dtype=MODEL_DTYPE, enabled=True):
        scores = forward_nemotron_scores(model, batch)
        logits = scores.view(num_groups, LISTWISE_GROUP_SIZE)
        loss = F.cross_entropy(logits.float(), positive_indices)

    loss.backward()

    peak_gb = torch.cuda.max_memory_allocated() / 1024**3

    del batch, positive_indices, scores, logits, loss
    model.zero_grad(set_to_none=True)
    cleanup_cuda()

    return peak_gb

def autotune_train_group_batch_size():
    print("\nAutotuning train group batch size with gradient checkpointing enabled...")

    successful = []

    for bs in TRAIN_GROUP_BATCH_SIZE_CANDIDATES:
        try:
            cleanup_cuda()
            print(f"\nTrying train group batch size = {bs} ({bs * LISTWISE_GROUP_SIZE} pairs/forward)")
            peak_gb = try_train_microbatch(
                model=model,
                collator=base_collator,
                groups=train_groups,
                bs=bs,
            )
            print(f"Success bs={bs} | peak allocated during probe: {peak_gb:.2f}GB")
            successful.append((bs, peak_gb))

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"OOM at train group batch size {bs}. Trying smaller...")
                cleanup_cuda()
            else:
                raise

    if not successful:
        raise RuntimeError("No train batch size fit, even bs=1.")

    # Pick largest successful batch.
    chosen_bs, chosen_peak = successful[0]

    # If a larger batch technically fits but is too close to limit,
    # it may OOM after AdamW states are allocated. Prefer safety.
    # A100 40GB has ~39.5GB usable; keep a rough margin.
    SAFE_PEAK_LIMIT_GB = 28.0

    safe_successful = [(bs, peak) for bs, peak in successful if peak <= SAFE_PEAK_LIMIT_GB]

    if safe_successful:
        chosen_bs, chosen_peak = safe_successful[0]
        print(f"\nChosen safe train group batch size = {chosen_bs} | probe peak={chosen_peak:.2f}GB")
    else:
        # If none meet the safety threshold, use smallest successful.
        chosen_bs, chosen_peak = successful[-1]
        print(
            f"\nWARNING: no batch size met SAFE_PEAK_LIMIT_GB={SAFE_PEAK_LIMIT_GB}. "
            f"Using smallest successful bs={chosen_bs} | peak={chosen_peak:.2f}GB"
        )

    return chosen_bs

if AUTOTUNE_TRAIN_BATCH_SIZE:
    TRAIN_GROUP_BATCH_SIZE = autotune_train_group_batch_size()
else:
    TRAIN_GROUP_BATCH_SIZE = 1

GRAD_ACCUM_STEPS = max(1, math.ceil(TARGET_EFFECTIVE_GROUPS_PER_UPDATE / TRAIN_GROUP_BATCH_SIZE))

print("\nFinal batch configuration:")
print("TRAIN_GROUP_BATCH_SIZE:", TRAIN_GROUP_BATCH_SIZE)
print("GRAD_ACCUM_STEPS:", GRAD_ACCUM_STEPS)
print("Effective groups/update:", TRAIN_GROUP_BATCH_SIZE * GRAD_ACCUM_STEPS)
print("Pairs/forward:", TRAIN_GROUP_BATCH_SIZE * LISTWISE_GROUP_SIZE)

train_collator = base_collator

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_GROUP_BATCH_SIZE,
    shuffle=True,
    collate_fn=train_collator,
    num_workers=0,
    pin_memory=False,
)

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
max_train_steps = num_update_steps_per_epoch * MAX_EPOCHS
warmup_steps = int(max_train_steps * WARMUP_RATIO)

print("\nTraining schedule:")
print("Train groups:", len(train_dataset))
print("Optimizer steps per epoch:", num_update_steps_per_epoch)
print("Max epochs:", MAX_EPOCHS)
print("Max optimizer steps:", max_train_steps)
print("Warmup steps:", warmup_steps)


# ==========================================
# 14. OPTIMIZER + SCHEDULER
# ==========================================

def make_optimizer(model):
    params = [p for p in model.parameters() if p.requires_grad]
    try:
        print("Using fused AdamW.")
        return torch.optim.AdamW(
            params,
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            fused=True,
        )
    except TypeError:
        print("Fused AdamW unavailable; using standard AdamW.")
        return torch.optim.AdamW(
            params,
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )

optimizer = make_optimizer(model)

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=max_train_steps,
)


# ==========================================
# 15. EVAL FUNCTIONS
# ==========================================

def evaluate_scored_samples(scored_samples_by_lang, alpha=None):
    metrics = {}
    per_lang = {}

    for lang in LANGUAGES:
        samples = scored_samples_by_lang[lang]
        ranks = []

        for sample in samples:
            gold_id = str(sample["gold_id"])
            candidate_ids = [str(x) for x in sample["candidate_ids"]]
            ce_scores = np.asarray(sample["ce_scores"], dtype=np.float32)

            if alpha is None:
                final_scores = ce_scores
            else:
                dense_scores = np.asarray(sample["dense_scores"], dtype=np.float32)
                final_scores = alpha * zscore(dense_scores) + (1.0 - alpha) * zscore(ce_scores)

            order = np.argsort(-final_scores)
            reranked_ids = [candidate_ids[i] for i in order]
            rank = reranked_ids.index(gold_id) + 1 if gold_id in reranked_ids else 10000
            ranks.append(rank)

        lang_scores = compute_metrics_from_ranks(ranks)
        per_lang[lang] = lang_scores

        metrics[f"{lang}_mrr@1"] = lang_scores["MRR@1"]
        metrics[f"{lang}_mrr@5"] = lang_scores["MRR@5"]
        metrics[f"{lang}_mrr@10"] = lang_scores["MRR@10"]
        metrics[f"{lang}_recall@5"] = lang_scores["Recall@5"]
        metrics[f"{lang}_recall@10"] = lang_scores["Recall@10"]

    metrics["multilingual_avg_mrr@1"] = float(np.mean([per_lang[l]["MRR@1"] for l in LANGUAGES]))
    metrics["multilingual_avg_mrr@5"] = float(np.mean([per_lang[l]["MRR@5"] for l in LANGUAGES]))
    metrics["multilingual_avg_mrr@10"] = float(np.mean([per_lang[l]["MRR@10"] for l in LANGUAGES]))
    metrics["multilingual_avg_recall@5"] = float(np.mean([per_lang[l]["Recall@5"] for l in LANGUAGES]))
    metrics["multilingual_avg_recall@10"] = float(np.mean([per_lang[l]["Recall@10"] for l in LANGUAGES]))

    return metrics

def score_dev_candidates(model, epoch_label):
    scored = {}

    print(f"\nScoring dev candidates with Nemotron full FT | {epoch_label}")
    model.eval()

    for lang in LANGUAGES:
        partial_path = os.path.join(EVAL_DIR, f"dev_scored_{epoch_label}_top{TOPK}_{lang}_partial.json.gz")

        if os.path.isfile(partial_path):
            print(f"Loading existing partial {lang.upper()}:", partial_path)
            partial = load_json_gz(partial_path)
            scored[lang] = partial[lang]
            continue

        samples = dev_candidates_by_lang[lang]

        flat_pairs = []
        counts = []

        for sample in samples:
            q = sample["query"]
            docs = sample["candidate_texts"]
            counts.append(len(docs))
            flat_pairs.extend((q, d) for d in docs)

        flat_scores, used_bs = score_flat_pairs_with_fallback(
            model=model,
            tokenizer=tokenizer,
            pairs=flat_pairs,
            device=device,
            desc_prefix=f"Score dev {lang.upper()} | {epoch_label}",
        )

        scored_samples = []
        offset = 0

        for sample, count in zip(samples, counts):
            new_sample = dict(sample)
            new_sample["ce_scores"] = flat_scores[offset:offset + count].tolist()
            scored_samples.append(new_sample)
            offset += count

        scored[lang] = scored_samples

        save_json_gz(partial_path, {lang: scored_samples})
        print(f"Saved partial {lang.upper()} with eval batch {used_bs}:", partial_path)

    return scored

def run_alpha_sweep(scored_samples_by_lang, title, save_prefix):
    print(f"\n===== ALPHA SWEEP | {title} =====")

    pure_metrics = evaluate_scored_samples(scored_samples_by_lang, alpha=None)
    print_multilingual_metrics(f"{title} | pure reranker", pure_metrics)

    alpha_summary = []
    alpha_results = {}

    for alpha in FUSION_ALPHAS:
        metrics = evaluate_scored_samples(scored_samples_by_lang, alpha=alpha)
        key = f"{alpha:.3f}"
        alpha_results[key] = metrics

        row = {
            "alpha": float(alpha),
            "multilingual_avg_mrr@1": metrics["multilingual_avg_mrr@1"],
            "multilingual_avg_mrr@5": metrics["multilingual_avg_mrr@5"],
            "multilingual_avg_mrr@10": metrics["multilingual_avg_mrr@10"],
            "multilingual_avg_recall@5": metrics["multilingual_avg_recall@5"],
            "multilingual_avg_recall@10": metrics["multilingual_avg_recall@10"],
        }
        alpha_summary.append(row)

        print(
            f"alpha={alpha:.3f} | "
            f"MRR@5={metrics['multilingual_avg_mrr@5']:.4f} | "
            f"MRR@10={metrics['multilingual_avg_mrr@10']:.4f} | "
            f"Recall@5={metrics['multilingual_avg_recall@5']:.4f} | "
            f"Recall@10={metrics['multilingual_avg_recall@10']:.4f}"
        )

    best_by_mrr5 = max(alpha_summary, key=lambda x: x["multilingual_avg_mrr@5"])

    print(f"\n===== BEST ALPHA | {title} =====")
    for k, v in best_by_mrr5.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

    out = {
        "title": title,
        "pure_reranker_metrics": pure_metrics,
        "alpha_results": alpha_results,
        "alpha_summary": alpha_summary,
        "best_by_mrr5": best_by_mrr5,
    }

    save_json(os.path.join(EVAL_DIR, f"{save_prefix}_alpha_sweep.json"), out)
    return out

def save_full_model_checkpoint(path, model, tokenizer, extra_meta=None):
    os.makedirs(path, exist_ok=True)

    model.save_pretrained(
        path,
        safe_serialization=True,
        max_shard_size="2GB",
    )
    tokenizer.save_pretrained(path)

    if extra_meta is not None:
        save_json(os.path.join(path, "ct26_checkpoint_meta.json"), extra_meta)

    print("Saved full model checkpoint:", path)


# Optional pretrain eval.
if RUN_PRETRAIN_EVAL:
    pre_scored = score_dev_candidates(model, "pretrain")
    save_json_gz(os.path.join(EVAL_DIR, f"dev_scored_pretrain_top{TOPK}.json.gz"), pre_scored)
    pre_eval = run_alpha_sweep(pre_scored, "pretrain", "pretrain")


# ==========================================
# 16. TRAIN LOOP — FULL FINE-TUNING
# ==========================================

print("\nStarting Nemotron 1B FULL fine-tuning...")

best_metric = -1.0
best_epoch = None
best_alpha = None
best_eval_payload = None
history = []
bad_epochs = 0

global_update_step = 0
raw_forward_step = 0
optimizer.zero_grad(set_to_none=True)

start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss_sum = 0.0
    epoch_loss_count = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS}")

    for batch_idx, batch in enumerate(progress, start=1):
        positive_indices = batch.pop("positive_indices").to(device)
        num_groups = int(batch.pop("num_groups"))

        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        with torch.autocast(device_type="cuda", dtype=MODEL_DTYPE, enabled=True):
            scores = forward_nemotron_scores(model=model, batch=batch)
            logits = scores.view(num_groups, LISTWISE_GROUP_SIZE)
            loss = F.cross_entropy(logits.float(), positive_indices)

        loss_value = float(loss.detach().cpu().item())
        epoch_loss_sum += loss_value
        epoch_loss_count += 1

        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()

        raw_forward_step += 1

        should_step = (batch_idx % GRAD_ACCUM_STEPS == 0) or (batch_idx == len(train_loader))

        if should_step:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_update_step += 1

            if SAVE_EVERY_OPT_STEPS and global_update_step % SAVE_EVERY_OPT_STEPS == 0:
                ckpt_dir = os.path.join(CHECKPOINT_DIR, f"checkpoint-step{global_update_step}")
                save_full_model_checkpoint(
                    ckpt_dir,
                    model,
                    tokenizer,
                    extra_meta={
                        "epoch": epoch,
                        "global_update_step": global_update_step,
                        "raw_forward_step": raw_forward_step,
                        "loss_recent": loss_value,
                        "base_model": NEMOTRON_MODEL_NAME,
                        "stage2_root": STAGE2_ROOT,
                    },
                )

        if batch_idx % LOGGING_STEPS == 0 or batch_idx == len(train_loader):
            avg_loss = epoch_loss_sum / max(epoch_loss_count, 1)
            lr_now = scheduler.get_last_lr()[0]
            progress.set_postfix(
                loss=f"{avg_loss:.4f}",
                lr=f"{lr_now:.2e}",
                opt_step=global_update_step,
            )

        del batch, positive_indices, scores, logits, loss

        if batch_idx % 200 == 0:
            gc.collect()

    train_loss_avg = epoch_loss_sum / max(epoch_loss_count, 1)
    print(f"\nEpoch {epoch} train loss: {train_loss_avg:.6f}")

    # Save full epoch checkpoint BEFORE evaluation so an eval OOM never wastes training.
    epoch_ckpt_dir = os.path.join(CHECKPOINT_DIR, f"checkpoint-epoch{epoch}")
    save_full_model_checkpoint(
        epoch_ckpt_dir,
        model,
        tokenizer,
        extra_meta={
            "epoch": epoch,
            "global_update_step": global_update_step,
            "train_loss": train_loss_avg,
            "base_model": NEMOTRON_MODEL_NAME,
            "stage2_root": STAGE2_ROOT,
            "full_finetune": True,
            "gradient_checkpointing": True,
        },
    )

    cleanup_cuda()
    print_gpu_memory(f"after saving checkpoint epoch {epoch}")

    scored_epoch = score_dev_candidates(
        model=model,
        epoch_label=f"epoch{epoch}",
    )

    scored_epoch_path = os.path.join(EVAL_DIR, f"dev_scored_epoch{epoch}_top{TOPK}.json.gz")
    save_json_gz(scored_epoch_path, scored_epoch)
    print("Saved scored dev:", scored_epoch_path)

    eval_payload = run_alpha_sweep(
        scored_samples_by_lang=scored_epoch,
        title=f"epoch{epoch}",
        save_prefix=f"epoch{epoch}",
    )

    current_metric = eval_payload["best_by_mrr5"]["multilingual_avg_mrr@5"]
    current_alpha = eval_payload["best_by_mrr5"]["alpha"]

    history_row = {
        "epoch": epoch,
        "train_loss": train_loss_avg,
        "global_update_step": global_update_step,
        "scored_dev_path": scored_epoch_path,
        "best_alpha": current_alpha,
        "best_multilingual_avg_mrr@5": current_metric,
        "best_multilingual_avg_mrr@10": eval_payload["best_by_mrr5"]["multilingual_avg_mrr@10"],
        "best_multilingual_avg_recall@5": eval_payload["best_by_mrr5"]["multilingual_avg_recall@5"],
        "best_multilingual_avg_recall@10": eval_payload["best_by_mrr5"]["multilingual_avg_recall@10"],
    }
    history.append(history_row)
    save_json(TRAINING_HISTORY_PATH, history)

    improved = current_metric > best_metric + 1e-8

    if improved:
        best_metric = current_metric
        best_epoch = epoch
        best_alpha = current_alpha
        best_eval_payload = eval_payload
        bad_epochs = 0

        if os.path.isdir(BEST_RERANKER_DIR):
            shutil.rmtree(BEST_RERANKER_DIR)

        # Copy epoch checkpoint to best dir instead of saving full weights again.
        shutil.copytree(epoch_ckpt_dir, BEST_RERANKER_DIR)

        save_json(
            os.path.join(BEST_RERANKER_DIR, "ct26_checkpoint_meta.json"),
            {
                "best_epoch": best_epoch,
                "best_alpha": best_alpha,
                "best_multilingual_avg_mrr@5": best_metric,
                "base_model": NEMOTRON_MODEL_NAME,
                "loss_type": "listwise_softmax_ce_sequence_classification_logit",
                "score_type": "sequence_classification_raw_logit",
                "topk": TOPK,
                "stage2_root": STAGE2_ROOT,
                "retriever_root": SHARED_RETRIEVER_ROOT,
                "full_finetune": True,
                "gradient_checkpointing": True,
                "train_group_batch_size": TRAIN_GROUP_BATCH_SIZE,
                "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
            },
        )

        save_json(FINAL_METRICS_PATH, best_eval_payload)
        save_json(FINAL_ALPHA_SUMMARY_PATH, best_eval_payload["alpha_summary"])

        print(
            f"\nNew best FULL FT Nemotron saved | epoch={best_epoch} | "
            f"alpha={best_alpha:.3f} | MRR@5={best_metric:.4f}"
        )

    else:
        bad_epochs += 1
        print(
            f"\nNo improvement this epoch. "
            f"bad_epochs={bad_epochs}/{EARLY_STOPPING_PATIENCE}"
        )

        if bad_epochs >= EARLY_STOPPING_PATIENCE:
            print(
                f"\nEarly stopping triggered after epoch {epoch}. "
                f"Best epoch={best_epoch}, best MRR@5={best_metric:.4f}"
            )
            break

elapsed_hours = (time.time() - start_time) / 3600.0

print("\nTraining complete.")
print("Elapsed hours:", round(elapsed_hours, 2))
print("Best epoch:", best_epoch)
print("Best alpha:", best_alpha)
print("Best multilingual avg MRR@5:", best_metric)
print("Best full FT model dir:", BEST_RERANKER_DIR)

if DELETE_NON_BEST_EPOCH_CHECKPOINTS:
    print("\nDeleting non-best epoch checkpoints to save Drive space...")
    for name in os.listdir(CHECKPOINT_DIR):
        path = os.path.join(CHECKPOINT_DIR, name)
        if os.path.isdir(path) and name.startswith("checkpoint-epoch"):
            # Best model has already been copied to BEST_RERANKER_DIR.
            shutil.rmtree(path)
            print("Deleted:", path)


# ==========================================
# 17. SAVE FINAL META
# ==========================================

meta = {
    "dataset_name": DATASET_NAME,
    "languages": LANGUAGES,

    "stage1_retriever": {
        "shared_retriever_root": SHARED_RETRIEVER_ROOT,
        "best_retriever_dir": BEST_RETRIEVER_DIR,
        "retriever_artifacts_dir": RETRIEVER_ARTIFACTS_DIR,
        "doc_embeddings": DOC_EMB_PATH,
        "doc_ids": DOC_IDS_PATH,
        "doc_raw": DOC_RAW_PATH,
        "retriever_task_instruction": RETRIEVER_TASK_INSTRUCTION,
    },

    "stage2_reranker": {
        "shared_reranker_root": SHARED_RERANKER_ROOT,
        "stage2_root": STAGE2_ROOT,
        "base_reranker_model": NEMOTRON_MODEL_NAME,
        "best_reranker_dir": BEST_RERANKER_DIR,
        "checkpoint_dir": CHECKPOINT_DIR,
        "cache_dir": CACHE_DIR,
        "eval_dir": EVAL_DIR,
        "nemotron_query_variant": NEMOTRON_QUERY_VARIANT,
        "score_type": "sequence_classification_raw_logit",
        "training_loss": "listwise_softmax_cross_entropy",
        "topk": TOPK,
        "listwise_group_size": LISTWISE_GROUP_SIZE,
        "nemotron_max_length": NEMOTRON_MAX_LENGTH,
        "full_finetune": True,
    },

    "training": {
        "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "train_group_batch_size": TRAIN_GROUP_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "effective_groups_per_update": TRAIN_GROUP_BATCH_SIZE * GRAD_ACCUM_STEPS,
        "pairs_per_forward": TRAIN_GROUP_BATCH_SIZE * LISTWISE_GROUP_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "warmup_steps": warmup_steps,
        "max_train_steps": max_train_steps,
        "actual_global_update_steps": global_update_step,
        "max_grad_norm": MAX_GRAD_NORM,
        "bf16": True,
        "flash_attention_2": USE_FLASH_ATTN,
        "gradient_checkpointing": True,
        "autotune_train_batch_size": AUTOTUNE_TRAIN_BATCH_SIZE,
    },

    "files": {
        "train_candidates": TRAIN_CANDIDATES_GZ,
        "dev_candidates": DEV_CANDIDATES_GZ,
        "train_groups": TRAIN_GROUPS_GZ,
        "training_history": TRAINING_HISTORY_PATH,
        "final_metrics": FINAL_METRICS_PATH,
        "final_alpha_summary": FINAL_ALPHA_SUMMARY_PATH,
        "meta": META_PATH,
    },

    "best_result": {
        "best_epoch": best_epoch,
        "best_alpha": best_alpha,
        "best_multilingual_avg_mrr@5": best_metric,
    },

    "notes_for_test_time": {
        "load_model_from": BEST_RERANKER_DIR,
        "retriever_model_dir": BEST_RETRIEVER_DIR,
        "retriever_doc_embeddings": DOC_EMB_PATH,
        "candidate_depth": TOPK,
        "fusion_alpha_to_use": best_alpha,
        "score_function": "forward AutoModelForSequenceClassification and use raw relevance logit",
    },
}

save_json(META_PATH, meta)

print("\nSaved final files:")
print("Best full FT Nemotron model:", BEST_RERANKER_DIR)
print("Train candidates:", TRAIN_CANDIDATES_GZ)
print("Dev candidates:", DEV_CANDIDATES_GZ)
print("Train groups:", TRAIN_GROUPS_GZ)
print("Training history:", TRAINING_HISTORY_PATH)
print("Final metrics:", FINAL_METRICS_PATH)
print("Final alpha summary:", FINAL_ALPHA_SUMMARY_PATH)
print("Meta:", META_PATH)

cleanup_cuda()
print("\nDone.")

Mounted at /content/drive
Retriever root: /content/drive/MyDrive/ct26_qwen3_embedding_8b_lora
Full FT Nemotron root: /content/drive/MyDrive/ct26_nemotron1b_full_finetune_listwise_top10
Best retriever dir: /content/drive/MyDrive/ct26_qwen3_embedding_8b_lora/best_qwen3_8b_lora_sentence_transformer
Retriever artifacts dir: /content/drive/MyDrive/ct26_qwen3_embedding_8b_lora/retrieval_artifacts
Best full FT model will save to: /content/drive/MyDrive/ct26_nemotron1b_full_finetune_listwise_top10/best_nemotron1b_full_finetuned
Optional existing Qwen top-10 cache dir: /content/drive/MyDrive/ct26_qwen3_reranker8b_lora_listwise_top10/cache

$ /usr/bin/python3 -m pip install -U "transformers>=4.51.0" "sentence-transformers>=5.0.0" "datasets>=2.19.0" "accelerate>=0.30.0" safetensors tqdm scikit-learn packaging ninja

$ /usr/bin/python3 -m pip install -U torchao --index-url https://download.pytorch.org/whl/cu128

$ /usr/bin/python3 -m pip install -U "https://github.com/lesj0610/flash-attention/rele

torch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
BF16: True
Stage-2 root: /content/drive/MyDrive/ct26_nemotron1b_full_finetune_listwise_top10
FlashAttention 2 available: True

Loading Qwen retriever artifacts...
Doc matrix shape: (10000, 4096)
Documents: 10000

Loading train/dev splits...


README.md: 0.00B [00:00, ?B/s]

en_train.json: 0.00B [00:00, ?B/s]

en_dev.json: 0.00B [00:00, ?B/s]

...

Saved partial EN with eval batch 128: /content/drive/MyDrive/ct26_nemotron1b_full_finetune_listwise_top10/eval/dev_scored_epoch3_top10_en_partial.json.gz
Trying eval batch size 128 for Score dev FR | epoch3


Score dev FR | epoch3 bs=128:   0%|          | 0/55 [00:00<?, ?it/s]

Saved partial FR with eval batch 128: /content/drive/MyDrive/ct26_nemotron1b_full_finetune_listwise_top10/eval/dev_scored_epoch3_top10_fr_partial.json.gz
Trying eval batch size 128 for Score dev DE | epoch3


Score dev DE | epoch3 bs=128:   0%|          | 0/31 [00:00<?, ?it/s]

Saved partial DE with eval batch 128: /content/drive/MyDrive/ct26_nemotron1b_full_finetune_listwise_top10/eval/dev_scored_epoch3_top10_de_partial.json.gz
Saved scored dev: /content/drive/MyDrive/ct26_nemotron1b_full_finetune_listwise_top10/eval/dev_scored_epoch3_top10.json.gz

===== ALPHA SWEEP | epoch3 =====

===== epoch3 | pure reranker =====
--- EN ---
  MRR@1     : 0.6556
  MRR@5     : 0.7138
  MRR@10    : 0.7202
  Recall@5  : 0.8046
  Recall@10 : 0.8510
--- FR ---
  MRR@1     : 0.7151
  MRR@5     : 0.7690
  MRR@10    : 0.7718
  Recall@5  : 0.8433
  Recall@10 : 0.8632
--- DE ---
  MRR@1     : 0.6114
  MRR@5     : 0.6658
  MRR@10    : 0.6721
  Recall@5  : 0.7435
  Recall@10 : 0.7902
===== MULTILINGUAL AVG =====
  MRR@1     : 0.6607
  MRR@5     : 0.7162
  MRR@10    : 0.7214
  Recall@5  : 0.7971
  Recall@10 : 0.8348
alpha=0.000 | MRR@5=0.7162 | MRR@10=0.7214 | Recall@5=0.7971 | Recall@10=0.8348
alpha=0.025 | MRR@5=0.7171 | MRR@10=0.7220 | Recall@5=0.7992 | Recall@10=0.8348
alpha=0.050